In [40]:
import requests
import os

import os
from mistralai.client import Mistral

from transformers import VitsModel, AutoTokenizer
import torch
from IPython.display import Audio, display

from faster_whisper import WhisperModel
from faster_whisper.audio import decode_audio

GOOGLE_TRANSLATE_API_KEY = os.environ["GOOGLE_TRANSLATE_API_KEY"]
MISTRAL_API_KEY = os.environ["MISTRAL_API_KEY"]


In [39]:
model = WhisperModel("dyeto/kanbior-fon-asr-v1-ct2", device="cpu", compute_type="int8")  

[2026-11-24 22:40:04.073] [ctranslate2] [thread 164797] [warning] Tip: experimental MPS backend is available and often 2x+ faster than CPU on Apple Silicon (varies by workload and hardware). Try device="mps".


In [67]:
def transcribe(path, vad=True, audio=None):
    if audio is None:
        audio = decode_audio(str(path), sampling_rate=16000)   # float32 mono, 16 kHz
    segments, _ = model.transcribe(audio, language="fr", beam_size=1, without_timestamps=True, vad_filter=vad)
    return audio, " ".join(s.text.strip() for s in segments)


In [2]:
def translate(text, source="fon", target="fr"):
    response = requests.post(
        "https://translation.googleapis.com/language/translate/v2",
        params={"key": GOOGLE_TRANSLATE_API_KEY},
        json={
            "q": text,
            "source": source,
            "target": target,
            "format": "text"
        },
        timeout=30
    )

    response.raise_for_status()
    return response.json()["data"]["translations"][0]["translatedText"]

In [30]:
SYSTEM_PROMPT = """
You are Kanbior, an AI assistant for Fon-speaking users.

You receive two versions of the user's message:
- The original message in Fon.
- Its automatic translation into French.

Use both versions to understand the user's intent.
The French translation may contain errors.

Answer the user's question in French only.
Provide a clear, natural and concise response suitable
for subsequent translation into Fon and speech synthesis.

If the user's intent is unclear, ask for clarification in French.

Nerver anwser too long. Keep your response short and to the point. No more than 3 sentences. No lists. No bullet points. No numbered lists. No code blocks. No tables. No emojis. No hashtags. No links.
"""

In [31]:
client = Mistral(api_key=MISTRAL_API_KEY)

def ask(fon_text: str, fr_text: str) -> str:
    response = client.chat.complete(
        model="mistral-small-latest",
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": f"""
Original (Fon): {fon_text}
Translation (French): {fr_text}
"""
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

In [49]:


model_tts = VitsModel.from_pretrained("facebook/mms-tts-fon")
tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-fon")

Loading weights: 100%|██████████| 762/762 [00:00<00:00, 4850.54it/s]


In [50]:
def synthesize_speech(text: str):
    inputs = tokenizer(text, return_tensors="pt")

    with torch.no_grad():
        output = model_tts(**inputs).waveform
        

    return Audio(output, rate=model_tts.config.sampling_rate)

In [44]:
path = "kanbior-asr/real-data/WhatsAppAudio5.opus"
audio, fon_text = transcribe(path)
display(Audio(audio, rate=16000, autoplay=False))
print(fon_text)

Étɛ́ nɔví towe bló nú we?


In [45]:
fr_text = translate(fon_text, source="fon", target="fr")
fr_text

"Qu'est-ce que ton frère t'a fait ?"

In [46]:
res_fr_text = ask(fon_text, fr_text)
res_fr_text

"Qu'est-ce que ton frère t'a fait ? Ou bien veux-tu savoir ce qu'il a dit ou donné ? Précise ta question."

In [47]:
res_fon_text = translate(res_fr_text, source="fr", target="fon")
res_fon_text

'Etɛ nɔví towe ka wà nú we? Alǒ, a jló na tuùn nǔ e é ɖɔ alǒ na é à? Mi kɛnklɛn bo tín nǔkanbyɔ mitɔn mɛ.'

In [51]:
display(synthesize_speech(res_fon_text))

In [66]:
def kanbior(path: str):
    c=[]
    
    audio, fon_text = transcribe(path)
    print("Input audio:")
    display(Audio(audio, rate=16000, autoplay=False))
    
    c.append(fon_text)
    fr_text = translate(fon_text, source="fon", target="fr")
    c.append(fr_text)
    
    res_fr_text = ask(fon_text, fr_text)
    c.append(res_fr_text)
    
    
    res_fon_text = translate(res_fr_text, source="fr", target="fon")
    c.append(res_fon_text)
    
    print("Output audio:")
    display(synthesize_speech(res_fon_text))
    return c

In [83]:
#build fon audio from french question to test the whole flow
question_fr = "Comment créer mon CV ?"
question_fon = translate(question_fr, source="fr", target="fon")
print(f"Fon question: {question_fon}")
audion_fon = synthesize_speech(question_fon)
##save the fon audio to a file
with open("question_fon.wav", "wb") as f:
    f.write(audion_fon.data)

Fon question: Nε un ka sixu blo CV ce gbɔn ?


In [84]:
path = "kanbior-asr/real-data/WhatsAppAudio9.opus"
path = "question_fon.wav"
audio = decode_audio(str(path), sampling_rate=16000)
display(Audio(audio, rate=16000, autoplay=False))

In [85]:

c=kanbior(path)
c

Input audio:


Output audio:


['Un ká sixú bló cí ce gbɔn.',
 'Mais je peux les fabriquer moi-même.',
 "D'accord, vous parlez de fabrication personnelle. Vous voulez savoir comment faire quelque chose vous-même ?",
 'E nyɔ, a ɖo xo ɖɔ wɛ do nuɖiɖe mɛɖesunɔ tɔn wu. A jló na tuùn lee a na bló nǔ ɖé gbɔn é hwiɖesunɔ à?']